In [27]:
# -----------------------------------------------------------------
# INTRODUCTION TO MONTE CARLO WITH PYTHON
# May 2018 - Feb 2023
# Example program by Jordi Faraudo
# Simulation of a sequence of configurations for the 2D Ising model
# Calculation of Energy and magnetization of a configuration
# -----------------------------------------------------------------
#
# The algorithm is based on Rajesh Singh (Cambridge University) blog with python resources in Physics.
# https://rajeshrinet.github.io/blog/2014/ising-model/
# Adapted by Jordi Faraudo 2018 for teaching purposes
#
# Here we import the numpy mathematical library and the plots library
# as in the other examples in the course
#
import numpy as np
from numpy.random import rand
import matplotlib.pyplot as plt

#
# Function with the interactions of the model (2D spin Ising model)
# and the solution method (Metropolis Monte Carlo)
#
def mcmove(config, N, beta,camp_aplicat, relacio_Eexch_Ecamp):
        #Loop with a size equal to spins in the system
        for i in range(N):
            for j in range(N):
                    #pick a random spin: generate integer random number between 0 and N
                    a = np.random.randint(0, N)
                    b = np.random.randint(0, N)
                    ## state of spin
                    s =  config[a, b]

                    #Interaccio primers veins
                    #calculate energy cost of flipping that spin (the % is for calculation of periodic boundary condition)
                    nb = config[(a+1)%N,b] + config[a,(b+1)%N] + config[(a-1)%N,b] + config[a,(b-1)%N]
                    cost = 2*s*nb
                    # canviaria l'spin?
                    #flip spin or not depending on the cost and its Boltzmann factor
                    ## (acceptance probability is given by Boltzmann factor with beta = 1/kBT)
                    if cost < 0:
                        flip_inter_veins = True
                    elif rand() < np.exp(-cost*beta):
                        flip_inter_veins = True
                    else:
                        flip_inter_veins = False

                    # Two states
                    #calculate energy cost of fliping current state
                    if (camp_aplicat <0 and s <0) or (camp_aplicat >0 and s >0):
                       #cost of exciting the system
                       cost=1.0
                    else:
                       #cost of decay
                       cost=-1.0
                    # canviaria l'spin?
                    #flip spin or not depending on the cost and its Boltzmann factor
                    ## (acceptance probability is given by Boltzmann factor with beta = 1/kBT)
                    if cost < 0:
                        flip_two_states = True
                    elif rand() < np.exp(-cost*beta):
                        flip_two_states = True
                    else:
                        flip_two_states = False

                    # Comparacio i decisio final
                    if flip_inter_veins == flip_two_states == True:
                      s= s*(-1)
                    elif flip_inter_veins == flip_two_states == False:
                      s= s
                    else:
                      # comparo les energies la menor es que decideix si canvia o no
                      if (relacio_Eexch_Ecamp >1):
                        if flip_inter_veins == True:
                          s= s*(-1)
                        elif flip_inter_veins == False:
                          s= s
                      else:
                        if flip_two_states == True:
                          s= s*(-1)
                        elif flip_two_states == False:
                          s=s
                    config[a, b] = s

        # return the new configuration
        return config

#This function makes an image of the spin configurations
def configPlot(f, config, i, N):
        ''' This modules plts the configuration '''
        X, Y = np.meshgrid(range(N), range(N))
        plt.pcolormesh(X, Y, config, vmin=-1.0, vmax=1.0, cmap='RdBu_r');
        plt.title('MC iteration=%d'%i);
        plt.axis('tight')
        plt.pause(0.1)

#This function calculates the energy of a given configuration for the plots of Energy as a function of T
def calcEnergy(config):
    '''Energy of a given configuration'''
    energy = 0
    for i in range(len(config)):
        for j in range(len(config)):
            S = config[i,j]
            nb = config[(i+1)%N, j] + config[i,(j+1)%N] + config[(i-1)%N, j] + config[i,(j-1)%N]
            energy += -nb*S
    return energy/4.

#This function calculates the magnetization of a given configuration
def calcMag(config):
    '''Magnetization of a given configuration'''
    mag = np.sum(config)
    return mag
#
# MAIN PROGRAM
# Here we set initial conditions and control the flow of the simulation
#
#size of the lattice
N = 64

# constants (Ferro BCC)
J= 10**(-21) # factor J en julios
magn_Bohr= 9.2740100657*10**(-24)  # magneto de Bohr en Julios * 1/Teslas
mom_mag = 2.22*magn_Bohr   # a Temp. ambiente o bajas Temp (0K o 2K)
nu_0= 1.25663706212*10**(-6)  # permeabilitat al vuit en H/m

#Enter data for the simulation
Temperatura =float(input("\n Please enter temperature in kelvin: "))
temp = (1.380649*10**(-23)*Temperatura)/J
camp_aplicat= float(input("\n Camp aplicat en Teslas (no 0 siusplau): "))
msrmnt = int(input("\n Enter number of Monte Carlo iterations (suggestion 1000):"))

# relacio entre energies per fer els flip
Energia_salt_camp= 2*nu_0*mom_mag*camp_aplicat   # energia per passar d'spin igual al camp a oposat
Energia_salt_exchange = 2*J # energia per passar d'spins iguals a oposats
relacio_Eexch_Ecamp = Energia_salt_exchange / Energia_salt_camp

#Init Magnetization and Energy
step=[]
M=[]
E=[]

#Generate initial condition
config = 2*np.random.randint(2, size=(N,N))-1

#Calculate initial value of magnetization and Energy
Ene = calcEnergy(config)/(N*N)     # calculate average energy
Mag = calcMag(config)/(N*N)        # calculate average magnetisation
t=0
print('MC step=',t,' Energy=',Ene,' M=',Mag)
#Update
step.append(t)
E.append(Ene)
M.append(Mag)

#Show initial condition
print('Initial configuration:')
print(config)
#f = plt.figure(figsize=(15, 15), dpi=80);
f = plt.figure(dpi=100)
configPlot(f, config, 0, N)
plt.show()

#Turn on interactive mode for plots
print("Starting MC simulation")
plt.ion()

#Perform the MC iterations
for i in range(msrmnt):
            #call MC calculation
            mcmove(config, N, 1.0/temp, camp_aplicat, relacio_Eexch_Ecamp)
            #update variables
            t=t+1                              # update MC step
            Ene = calcEnergy(config)/(N*N)     # calculate average energy
            Mag = calcMag(config)/(N*N)        # calculate average magnetisation
            #Update
            step.append(t)
            E.append(Ene)
            M.append(Mag)

            #plot only certain configurations
            if t%10 == 0:
                print('\nMC step=',t,' Energy=',Ene,' M=',Mag)
                print(config)
                configPlot(f, config, t, N)

#Print end
print('\nSimulation finished after',t, 'MC steps')

#interactive plotting off
plt.ioff()

#Show final configuration
configPlot(f, config, t, N)
plt.show()

#Plot evolution of Energy and Magnetization during the simulation
plt.subplot(2, 1, 1)
plt.plot(step, E, 'r+-')
plt.ylabel('Energy')

plt.subplot(2, 1, 2)
plt.plot(step, M, 'b+-')
plt.ylabel('Magnetization')
plt.xlabel('MC step')

#Show the plot in screen
plt.show()


Output hidden; open in https://colab.research.google.com to view.